In [1]:
from comet_ml import API
import pandas as pd
import re
import os
import numpy as np

api_key = os.getenv("COMETML_API_KEY", "")
workspace = os.getenv("COMETML_WORKSPACE", "")
project_name = "sampling-bench"

api = API(api_key=api_key)
experiments = api.get_experiments(workspace=workspace, project_name=project_name)


In [13]:
def filter_experiments(experiments, filter_name):
    filtered_experiments = []
    for experiment in experiments:
        if filter_name in experiment.name:
            filtered_experiments.append(experiment)
    return filtered_experiments

def get_target_sd(experiment):
    for metric in experiment.get_metrics():
        if metric["metricName"] == "discrepancies/sd_target":
            return float(metric["metricValue"])

def get_metrics(experiment, filter_name, max_step):
    metric_name_to_table_name = {"KL/eubo": "EUBO", "KL/elbo": "ELBO", "logZ/delta_reverse": "$\Delta \log Z$", "Z/delta_reverse": "$\Delta Z$", "discrepancies/sd": "SD"}
    result = {}
    last_updated_step = {}
    for metric in experiment.get_metrics():
        if metric["metricName"] not in metric_name_to_table_name.keys():
            continue
        if metric["step"] is None or (metric["step"] <= max_step and last_updated_step.get(metric["metricName"], 0) <= metric["step"]):
            result[metric_name_to_table_name[metric["metricName"]]] = metric["metricValue"]
            last_updated_step[metric["metricName"]] = metric["step"]
    print("\t", experiment.name)
    for name, step in last_updated_step.items():
        print("\t\t", name, " step ", step)
    algorithm_name = experiment.name.split(filter_name)[0][:-1]
    algorithm_name = algorithm_name.replace("_", "\_")
    if "no_lp" in experiment.name:
        algorithm_name = f"{algorithm_name} (w/o LP)"
    result["algorithm_name"] = "$\\text{" + algorithm_name + "}$"
    return result

def create_latex_table(experiments, filter_name, gt_log_Z):
    # Collect all metric dicts with algorithm name as one entry
    rows = []
    for experiment in experiments:
        row = get_metrics(experiment, filter_name, 20000)
        rows.append(row)

    target_sd = get_target_sd(experiments[0])

    df = pd.DataFrame(rows)
    df = df.set_index("algorithm_name").rename_axis("Algorithm")
    df = df.replace("NaN", "-").fillna("-")
    df = df[sorted(df.columns)]
    df = df.sort_index()

    for col in df.columns:
        try:
            vals = pd.to_numeric(df[col], errors="coerce")
        except Exception:
            continue
        if vals.notna().any():
            closest_idx = None
            if col in ["ELBO", "EUBO"]:
                gt = gt_log_Z
            else:
                gt = 0.0
            try:
                closest_idx = (np.abs(vals - gt)).idxmin()
            except Exception:
                continue
            df[col] = df[col].apply(lambda x: str(round(x, 3)) if isinstance(x, (float, int)) else (x if x == "-" else str(round(float(x),3))))
            if closest_idx is not None:
                val = df.at[closest_idx, col]
                df.at[closest_idx, col] = f"\\textbf{{{val}}}"

    latex_table = df.to_latex(
        escape=False,
        float_format=None,
        label=f"table:{filter_name}",
        caption=None,
    )
    env_name = filter_name.replace("_", "\_")
    caption_text = f"{env_name}. Target SD {target_sd:.3f}."
    if gt_log_Z != 0.0:
        caption_text = f"{caption_text} Target $\log Z$ {gt_log_Z:.3f}."
    caption_text = "\\caption{" + caption_text + "}\n"
    latex_table = latex_table.replace("\\begin{table}", "\\begin{table}[!h]\n\\centering")
    latex_table = latex_table.replace("\\end{table}", caption_text + "\\end{table}")
    
    return latex_table

In [14]:
# funnel_10, many_well_32, student_t_mixture_50d, gaussian_mixture40_5d
tables = []
log_Z_maps = {"funnel_10": 0.0, "many_well_32": 164.69568, "student_t_mixture_50d": 0.0, "gaussian_mixture40_5d": 0.0}
for filter_name in log_Z_maps.keys():
    print(f"Creating {filter_name} table")
    table = create_latex_table(filter_experiments(experiments, filter_name), filter_name, log_Z_maps[filter_name])
    tables.append(table)

with open("tables.txt", "w") as f:
    f.write("\n".join(tables))

Creating funnel_10 table
	 pis_funnel_10
		 KL/eubo  step  19999
		 Z/delta_reverse  step  19999
		 discrepancies/sd  step  19999
		 KL/elbo  step  19999
		 logZ/delta_reverse  step  19999
	 dds_funnel_10
		 KL/eubo  step  19999
		 Z/delta_reverse  step  19999
		 discrepancies/sd  step  19999
		 KL/elbo  step  19999
		 logZ/delta_reverse  step  19999
	 dis_funnel_10
		 KL/eubo  step  20000
		 Z/delta_reverse  step  20000
		 discrepancies/sd  step  20000
		 KL/elbo  step  20000
		 logZ/delta_reverse  step  20000
	 ais_funnel_10
		 Z/delta_reverse  step  None
		 KL/elbo  step  None
		 discrepancies/sd  step  None
		 logZ/delta_reverse  step  None
	 cmcd_funnel_10
		 KL/eubo  step  20000
		 Z/delta_reverse  step  20000
		 discrepancies/sd  step  20000
		 KL/elbo  step  20000
		 logZ/delta_reverse  step  20000
	 gfn_tb_funnel_10
		 KL/eubo  step  19999
		 Z/delta_reverse  step  19999
		 discrepancies/sd  step  19999
		 KL/elbo  step  19999
		 logZ/delta_reverse  step  19999
	 smc_funnel_10